# DeepDoc OCR-VL — toàn bộ tài liệu trên Colab GPU
**Chọn GPU runtime, điền đường dẫn trong ô 1, rồi Run all.**

Model được nạp **một lần cho cả tài liệu**. Trang xử lý xong được lưu ngay trên Drive.
Chạy lại cùng tài liệu/cấu hình sẽ tiếp tục từ các trang còn thiếu. Không cần thêm ô vá lỗi.
Kết quả là raw OCR evidence; chưa tự publish/index vào RAG và chưa chứng nhận độ chính xác nhận dạng.


In [5]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, hashlib, subprocess, time, signal, datetime

RAG_DRIVE = '/content/drive/MyDrive/rag'
INPUT_DRIVE = '/content/drive/MyDrive/book/ban_chinh_sua_final.pdf' # Set the actual Drive path if different.
PAGES = None # PDF page numbers; None runs the whole PDF.
RESUME = True # Tiếp tục các trang chưa hoàn thành khi chạy lại
DPI = 144
MAX_RENDER_PIXELS = 1_000_000
MAX_PIXELS = 1_000_000 # Trần ảnh của VLM; tăng có thể tăng VRAM/thời gian
MAX_NEW_TOKENS = 2048 # Trần generation; nội dung dài vẫn có thể bị cắt
LOAD_TIMEOUT_SECONDS = 900
PAGE_TIMEOUT_SECONDS = 600
GPU_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Chuẩn bị môi trường
Venv Linux đặt trong `/content`; pip cache và trọng số đã tải lưu trên Drive.
Lần đầu vẫn cần cài thư viện/tải model. Notebook kiểm tra tokenizer trước khi nạp model, gồm bản sửa SentencePiece/Protobuf.
Nếu input chưa được điền, ô này dừng ngay và chỉ rõ cấu hình cần sửa.


In [6]:
base=Path('/content/drive/MyDrive')
rag=Path(RAG_DRIVE).resolve()
assert (rag/'colab/ocr_vl_document.py').is_file(), f'Missing colab/ocr_vl_document.py under {rag}'
assert (rag/'colab/ocr_vl_document.py').is_file(), 'Chờ Drive đồng bộ cả colab/ocr_vl_document.py'
if not INPUT_DRIVE: raise ValueError('Điền INPUT_DRIVE tới FILE PDF/ảnh trong ô 1.')
source=Path(INPUT_DRIVE).resolve()
assert source.is_file(), f'Không thấy FILE input: {source}'
assert PAGES is None or (isinstance(PAGES,list) and PAGES and all(type(p) is int and p>0 for p in PAGES) and len(PAGES)==len(set(PAGES)))
assert 72<=DPI<=300 and 1<=MAX_RENDER_PIXELS<=4_000_000 and 1<=MAX_PIXELS<=4_000_000
assert 256<=MAX_NEW_TOKENS<=8192 and 30<=PAGE_TIMEOUT_SECONDS<=3600 and 30<=LOAD_TIMEOUT_SECONDS<=3600
gpu=subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'],capture_output=True,text=True)
if gpu.returncode: raise RuntimeError('Chọn GPU runtime. '+gpu.stderr)
print(gpu.stdout)
if 'cu126' in GPU_INDEX:
    driver=gpu.stdout.splitlines()[0].split(',')[1].strip()
    if tuple(map(int,driver.split('.'))) < (550,54,14): raise RuntimeError('Driver chưa phù hợp wheel cu126.')
work=Path('/content/deepdoc-colab'); work.mkdir(exist_ok=True)
cache=base/'deepdoc-colab-cache'; cache.mkdir(exist_ok=True)
os.environ['PIP_CACHE_DIR']=str(cache/'pip')
py=work/'venv/bin/python'
if not py.is_file():
    created=subprocess.run([sys.executable,'-m','venv',str(work/'venv')],capture_output=True,text=True)
    if created.returncode:
        bootstrap=work/'bootstrap'
        subprocess.run([sys.executable,'-m','pip','install','--target',str(bootstrap),'virtualenv'],check=True)
        subprocess.run([sys.executable,'-m','virtualenv',str(work/'venv')],env={**os.environ,'PYTHONPATH':str(bootstrap)},check=True)
req=['paddleocr[doc-parser]==3.7.0','paddlex==3.7.2','sentencepiece==0.2.1','protobuf==6.33.4','pymupdf','pillow']
stamp=hashlib.sha256(json.dumps([sys.version,GPU_INDEX,req,'paddlepaddle-gpu==3.3.1']).encode()).hexdigest()
stamp_path=work/'install-stamp.txt'
if not stamp_path.is_file() or stamp_path.read_text()!=stamp:
    subprocess.run([str(py),'-m','pip','install','paddlepaddle-gpu==3.3.1','--index-url',GPU_INDEX],check=True)
    subprocess.run([str(py),'-m','pip','install',*req],check=True)
    subprocess.run([str(py),'-m','pip','check'],check=True)
    stamp_path.write_text(stamp)
probe="from importlib.metadata import version; from sentencepiece import sentencepiece_model_pb2; from paddlex.inference.models.common.tokenizer.llama_tokenizer import LlamaTokenizer; import paddle; assert paddle.is_compiled_with_cuda(); assert paddle.device.cuda.device_count()>0; print('Tokenizer/CUDA PASS', version('sentencepiece'),version('protobuf'))"
subprocess.run([str(py),'-c',probe],check=True)
print('Môi trường sẵn sàng. Input:',source)


NVIDIA L4, 580.82.07, 23034 MiB

Môi trường sẵn sàng. Input: /content/drive/MyDrive/book/ban_chinh_sua_final.pdf


## 3. Chạy cả tài liệu
Một worker giữ model trong GPU xuyên suốt các trang. Cache model được staging vào `/content` trước khi chạy.
Lượt đầu có thể cần thời gian tải/nạp model; các trang sau không nạp lại model.
Nhấn Stop sẽ dừng worker; trang đã ghi thành công vẫn còn trên Drive để resume.
RAM đo là RSS worker; VRAM là tổng sử dụng trên GPU, lấy mẫu khoảng 2 giây nên không bảo đảm bắt được mọi peak.


In [7]:
def sync_cache(src,dst):
    src,dst=Path(src),Path(dst)
    if not src.exists(): return
    for p in src.rglob('*'):
        if not p.is_file() or p.name.endswith(('.lock','.incomplete','.tmp')): continue
        target=dst/p.relative_to(src)
        if target.is_file() and target.stat().st_size==p.stat().st_size and target.stat().st_mtime>=p.stat().st_mtime: continue
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,target)

local_home=work/'model-home'; local_home.mkdir(exist_ok=True)
print('Đưa cache model về ổ local...')
sync_cache(cache/'model-home/.paddlex',local_home/'.paddlex')
sync_cache(cache/'model-home/.cache',local_home/'.cache')
runner=work/'ocr_vl_document.py'; shutil.copy2(rag/'colab/ocr_vl_document.py',runner)
local_input=work/('document'+source.suffix.lower()); shutil.copy2(source,local_input)
attempt=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S')+'-'+os.urandom(3).hex()
result_root=base/'deepdoc-colab-results'; attempts=result_root/'attempts'/attempt; attempts.mkdir(parents=True)
progress=work/(attempt+'-progress.json')
config={'input':str(local_input),'output_root':str(result_root),'progress':str(progress),
        'local_work':str(work),'pages':PAGES,'resume':RESUME,
        'dpi':DPI,'max_render_pixels':MAX_RENDER_PIXELS,'max_pixels':MAX_PIXELS,'max_new_tokens':MAX_NEW_TOKENS}
config_path=work/(attempt+'-config.json'); config_path.write_text(json.dumps(config),encoding='utf-8')
shutil.copy2(config_path,attempts/'config.json'); shutil.copy2(runner,attempts/'ocr_vl_document.py')
(attempts/'environment.txt').write_text(subprocess.check_output([str(py),'-m','pip','freeze'],text=True),encoding='utf-8')
worker_env={**os.environ,'HOME':str(local_home),'HF_HUB_DISABLE_XET':'1','PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK':'True','OMP_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','TOKENIZERS_PARALLELISM':'false'}
log_path=attempts/'worker.log'; proc=None; samples=[]; latest={}; last_key=None; stage_started=time.monotonic(); last_print=0; failure=None
try:
    with log_path.open('w',encoding='utf-8') as log:
        proc=subprocess.Popen([str(py),str(runner),'--config',str(config_path)],env=worker_env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        while proc.poll() is None:
            if progress.is_file():
                try: latest=json.loads(progress.read_text(encoding='utf-8'))
                except (OSError,ValueError): pass
            key=(latest.get('stage'),latest.get('page'))
            if key!=last_key: stage_started=time.monotonic();last_key=key
            limit=LOAD_TIMEOUT_SECONDS if latest.get('stage') in (None,'LOADING_MODEL') else PAGE_TIMEOUT_SECONDS
            if time.monotonic()-stage_started>limit: raise TimeoutError(f'Vượt {limit}s tại {key}')
            rss=None
            try:
                rss=int(next(s for s in Path(f'/proc/{proc.pid}/status').read_text().splitlines() if s.startswith('VmRSS:')).split()[1])/1024
            except (OSError,StopIteration): pass
            mem=subprocess.run(['nvidia-smi','--query-gpu=memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=10)
            samples.append({'time':time.time(),'stage':key,'worker_rss_mib':rss,'gpu_used_total_utilization':mem.stdout.strip()})
            if time.monotonic()-last_print>10:
                print(latest.get('stage','STARTING'),'page',latest.get('page'),'completed',len(latest.get('completed',[])),'RAM MiB',rss,'GPU',mem.stdout.strip(),flush=True)
                last_print=time.monotonic()
            time.sleep(2)
        returncode=proc.wait()
        if progress.is_file(): latest=json.loads(progress.read_text(encoding='utf-8'))
        if returncode or latest.get('stage')!='SUCCESS': raise RuntimeError(f'Worker exit {returncode}; {latest.get("error", "chưa hoàn tất")}')
except BaseException as exc:
    failure=f'{type(exc).__name__}: {exc}'
finally:
    if proc is not None and proc.poll() is None:
        try: os.killpg(proc.pid,signal.SIGKILL)
        except ProcessLookupError: pass
        proc.wait()
    (attempts/'memory-samples.json').write_text(json.dumps(samples,ensure_ascii=False,indent=2),encoding='utf-8')
    (attempts/'summary.json').write_text(json.dumps({'error':failure,'returncode':proc.returncode if proc else None,'progress':latest},ensure_ascii=False,indent=2),encoding='utf-8')
    if failure:
        print(log_path.read_text(encoding='utf-8',errors='replace')[-12000:])
        print('Lỗi:',failure,'\nLog:',log_path)
    else:
        print('Lưu cache model vào Drive...')
        sync_cache(local_home/'.paddlex',cache/'model-home/.paddlex')
        sync_cache(local_home/'.cache',cache/'model-home/.cache')
if failure: raise RuntimeError(failure)
out=Path(latest['output'])
print('SUCCESS:',len(latest['completed']),'trang. Kết quả:',out)


Đưa cache model về ổ local...
STARTING page None completed 0 RAM MiB 0.19921875 GPU 0, 23034, 0
LOADING_MODEL page 1 completed 0 RAM MiB 1691.515625 GPU 8122, 23034, 0
INFERENCE page 2 completed 1 RAM MiB 2796.10546875 GPU 8168, 23034, 17
INFERENCE page 2 completed 1 RAM MiB 2796.10546875 GPU 8168, 23034, 17
INFERENCE page 3 completed 2 RAM MiB 2808.43359375 GPU 8828, 23034, 17
INFERENCE page 3 completed 2 RAM MiB 2808.43359375 GPU 8828, 23034, 18
INFERENCE page 3 completed 2 RAM MiB 2808.43359375 GPU 8828, 23034, 18
INFERENCE page 4 completed 3 RAM MiB 2806.9609375 GPU 8828, 23034, 16
INFERENCE page 5 completed 4 RAM MiB 2807.578125 GPU 8828, 23034, 16
INFERENCE page 5 completed 4 RAM MiB 2807.578125 GPU 8828, 23034, 17
INFERENCE page 5 completed 4 RAM MiB 2807.58203125 GPU 8828, 23034, 16
INFERENCE page 6 completed 5 RAM MiB 2807.5859375 GPU 8828, 23034, 17
INFERENCE page 6 completed 5 RAM MiB 2807.59765625 GPU 8828, 23034, 17
INFERENCE page 6 completed 5 RAM MiB 2807.59765625 GPU 88

In [8]:
document=json.loads((out/'document.json').read_text(encoding='utf-8'))
for page in document['pages']:
    result=json.loads((out/f'page-{page:04d}.json').read_text(encoding='utf-8'))
    print(f"Trang {page}: inference {result['seconds']:.2f}s; raw JSON đã lưu")
first=json.loads((out/f"page-{document['pages'][0]:04d}.json").read_text(encoding='utf-8'))
print(json.dumps(first['raw'],ensure_ascii=False,indent=2)[:5000])
print('Kết quả:',out,'\nLog/RAM/VRAM:',attempts)


Trang 1: inference 4.28s; raw JSON đã lưu
Trang 2: inference 19.30s; raw JSON đã lưu
Trang 3: inference 25.86s; raw JSON đã lưu
Trang 4: inference 12.75s; raw JSON đã lưu
Trang 5: inference 28.59s; raw JSON đã lưu
Trang 6: inference 28.52s; raw JSON đã lưu
Trang 7: inference 17.17s; raw JSON đã lưu
Trang 8: inference 18.19s; raw JSON đã lưu
Trang 9: inference 28.48s; raw JSON đã lưu
Trang 10: inference 28.05s; raw JSON đã lưu
Trang 11: inference 21.21s; raw JSON đã lưu
Trang 12: inference 25.03s; raw JSON đã lưu
Trang 13: inference 18.89s; raw JSON đã lưu
Trang 14: inference 25.20s; raw JSON đã lưu
Trang 15: inference 22.37s; raw JSON đã lưu
Trang 16: inference 26.06s; raw JSON đã lưu
Trang 17: inference 24.73s; raw JSON đã lưu
Trang 18: inference 24.62s; raw JSON đã lưu
Trang 19: inference 21.55s; raw JSON đã lưu
Trang 20: inference 1.00s; raw JSON đã lưu
Trang 21: inference 20.48s; raw JSON đã lưu
Trang 22: inference 27.84s; raw JSON đã lưu
Trang 23: inference 26.83s; raw JSON đã lưu

## Giới hạn và cách chạy tiếp
- Giữ `RESUME=True`, cùng input/cấu hình và Run all để bỏ qua trang có checkpoint hợp lệ. Thay file/cấu hình/runner/package sẽ tạo bộ kết quả riêng.
- Nếu Drive đồng bộ chưa có `colab/ocr_vl_document.py`, chờ đồng bộ trước khi chạy.
- Tốc độ phụ thuộc nội dung trang và GPU; không cam kết thời gian/trang khi chưa đo trên Colab. `SUCCESS` chứng minh tiến trình chạy xong, không bảo đảm OCR đủ/đúng mọi chữ. Giới hạn token có thể cắt nội dung dài.
- Venv Windows không được dùng. Phiên Colab mới phải tạo lại venv Linux; cache trên Drive giảm tải lại.
- Notebook đã được kiểm tra cấu trúc/cú pháp tại local; worker phiên bản này chưa được chạy trên GPU Colab lúc bàn giao.

[PaddleOCR-VL API](https://www.paddleocr.ai/main/en/version3.x/pipeline_usage/PaddleOCR-VL.html) · [GPU wheel](https://paddlepaddle.github.io/FastDeploy/get_started/installation/nvidia_gpu/) · [Colab FAQ](https://research.google.com/colaboratory/faq.html)
